批量调用文件内容，然后批量openai api查询，然后批量处理保存结果

In [233]:
import requests
from io import BytesIO
from openai import OpenAI
import os
from pathlib import Path
import json
import pandas as pd

In [234]:
def get_all_files_1(directory):
    file_list = []
    # os.walk 会递归查找子文件夹里的所有文件
    for root, dirs, files in os.walk(directory):
        for file in files:
            # 拼接完整路径
            full_path = os.path.join(root, file)
            file_list.append(full_path)
    return file_list

def get_all_files(directory):
    p = Path(directory)
    # 只取一级文件；只要 .json；排序保证稳定
    files = [
        str(x) for x in p.iterdir()
        if x.is_file() and x.suffix.lower() == ".json"
    ]
    return sorted(files)

def extract_urls_fingerprint(data, apk_name_from_filename: str):
    # data: dict, e.g. { "<apk>": [ {url:...}, ... ] }
    if not isinstance(data, dict) or not data:
        return tuple()

    # 通常顶层 key 就是 apkname；为鲁棒起见，不匹配就取第一个 key
    apk_key = apk_name_from_filename if apk_name_from_filename in data else next(iter(data.keys()))
    records = data.get(apk_key, [])
    if not isinstance(records, list):
        return tuple()

    urls = []
    for rec in records:
        if isinstance(rec, dict):
            u = rec.get("url")
            if isinstance(u, str) and u.strip():
                urls.append(u.strip())

    # fingerprint：排序后的唯一 url 列表（tuple 可 hash/可比）
    return tuple(sorted(set(urls)))

def parse_file_info(path):
    # 拿到文件名：ae.brandsforless.android_412_merged_privacy_url_evidence.json
    filename = os.path.basename(path)
    # 按照下划线切分
    parts = filename.split('_')
    apk_name = parts[0]   # ae.brandsforless.android
    version = parts[1]    # 412
    return apk_name, version

先拿第一批的second来跑，因为second恰好100个

得建好输出文档，输出目录也是批处理目录

In [235]:
# 加一个逻辑，一个apkname就选一个文档，不需要重复的apkname的文档了。可以在处理文件时维护一个集合，记录已经处理过的apkname，如果遇到重复的就跳过。示例代码如下：


# 加一个逻辑，一个apkname如果已经处理了，就跳过后续的同名apk文件。如何判断apkname的内容大差不大：如果target_dir同apkname的文件是json文件，如果它们里面的apkname.url内容一样，那么就认为它们内容大差不大，可以只处理第一个文件。示例代码如下：
# 同 apkname 的文件内容大差不大”的一个稳妥判定方式是：提取该文件里该 apk 的所有 url 去重后的集合做 fingerprint；如果同 apkname 的 fingerprint 一样，就跳过后续文件。
# target_dir = r'1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch_deduplicated' 

In [236]:
# {f_position}\{s_position}
# AA2_second_batch\AA6_sisth_100_batch f"_pevidence.json", \AA3_third_batch\AA3_third_100_batch f"_p.json"
f_position = "AA2_second_batch"
s_position = "AA7_seventh_100_batch" # AA7_seventh_100_batch # AA4_forth_100_batch, AA5_fifth_100_batch, AA6_sisth_100_batch
target_dir = rf'1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}_deduplicated'
target_dir

'1122apk\\1122apk_privacy_policy_url\\out_summary_json\\AA2_second_batch\\AA7_seventh_100_batch_deduplicated'

In [ ]:
results_container = []
seen_apk_fingerprint = {}

# jump_apkname_set = set() # 记录已经跳过的 apkname，方便 debug/log

client = OpenAI(api_key="")
# client = OpenAI(api_key="") 


# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch_deduplicated
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch_deduplicated
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch_deduplicated
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch_deduplicated
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch_deduplicated
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch_deduplicated

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
target_dir = rf'1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}_deduplicated' # r'1122apk\1122apk_privacy_policy_url\out_summary_json'
# leftover
# target_dir = r'1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch_deduplicated\leftover'

# output_dir = r'1122apk\1122apk_privacy_policy_url\out_openai_pp_txt_2024_10\AA1_first_batch\AA2_second_100_batch'
# AA1_first_328_batch
# AA2_second_100_batch
# AA3_third_100_batch
# AA4_forth_100_batch
# AA5_fifth_100_batch
# AA6_sisth_74_batch
output_dir = rf'1122apk\1122apk_privacy_policy_url\out_openai_pp_url\{f_position}\{s_position}'

output = Path(output_dir)
output.mkdir(parents=True, exist_ok=True)

all_files = get_all_files(target_dir)

count = 1

for file_path in all_files:
    
    # if count == 2:
    #      break
    
    apk_name, version = parse_file_info(file_path)
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            # 将 JSON 文件加载为 Python 对象
            data = json.load(f)
            
            # 将其转为字符串存入 file_content，方便发给 AI
            # indent=2 可以让格式更好看，AI 更容易阅读
            file_content = json.dumps(data, indent=2, ensure_ascii=False)
    except FileNotFoundError:
        # 出错也要记录，保证数据完整
        results_container.append({
            "apk_name": apk_name, "version": version,
            "privacy_url": None, "reasoning": f"Error: {str("file not found error")}"
        })
        continue

    # fp = extract_urls_fingerprint(data, apk_name)

    # # 去重规则：同 apk_name 且 url fingerprint 一样 => 跳过
    # if apk_name in seen_apk_fingerprint and fp == seen_apk_fingerprint[apk_name]:
    #     print(f"跳过 {apk_name}, {version}，因为 fingerprint 一样")
    #     jump_apkname_set.add((apk_name, version))
    #     continue

    # # 如果同名但 fingerprint 不一样：说明“url 内容不一样”，按你的描述应继续处理
    # # （也可以在这里 print/log 一下冲突）
    # if apk_name not in seen_apk_fingerprint:
    #     seen_apk_fingerprint[apk_name] = fp



    # 假设你已经读取了文件内容到 file_content 变量中
    prompt_content = f"""
    File Content:
    {file_content}

    Task: Identify the APK's privacy policy URL. 
    Exclude third-party SDKs. 

    Return ONLY a JSON object with this structure:
    {{
    "reason": "your reasoning process here",
    "result": "the url or null"
    }}
    If no URL is found, set "result" to null. Do not include any text outside the JSON.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-5.1", # 或者你使用的最新模型
            response_format={ "type": "json_object" },
            messages=[
                {
                    "role": "system", 
                    "content": "You are a specialized assistant that identifies privacy policy URLs in APK metadata. Look for first-party URLs and ignore third-party SDKs (like Unity, Tapjoy, etc.)."
                },
                {
                    "role": "user", 
                    "content": prompt_content
                }
            ]
        )
        # 读取结果
        print(response.choices[0].message.content)
        content_str = response.choices[0].message.content

        # 模拟 API 返回的合法 JSON 字符串
        # content_str = '{"reason": "Found in manifest", "result": "https://example.com/p"}'


        api_data = json.loads(content_str)
        
        # 3. 组装数据行
        row = {
            "apk_name": apk_name,
            "version": version,
            "privacy_url": api_data.get("result"), # 如果没有则为 None (JSON 的 null)
            "reasoning": api_data.get("reason"),
            "source_file": os.path.basename(file_path)  # 留个底，方便回溯
        }
        results_container.append(row)
        count = count + 1


    except Exception as e:
            print(f"处理 {apk_name} 出错: {e}")
            # 出错也要记录，保证数据完整
            results_container.append({
                "apk_name": apk_name, "version": version,
                "privacy_url": None, "reasoning": f"Error: {str(e)}"
            })

# print(f"总共有 {len(jump_apkname_set)} 个apk被跳过了，因为 fingerprint 一样。它们是：{jump_apkname_set}")

# 4. 一键转为 DataFrame
df = pd.DataFrame(results_container)

# 5. 保存结果
df.to_csv(f"{output_dir}/batch_privacy_results.csv", index=False, encoding='utf-8-sig')
# df.to_csv(f"{output_dir}/leftover_batch_privacy_results.csv", index=False, encoding='utf-8-sig')
print(df.head())

{
  "reason": "Most URLs reference third-party services or ad/analytics SDKs (ironSource, AppLovin, Firebase, Facebook, Google, Vungle, PubNative, Unity, AdColony, Fyber, InMobi, Mintegral, Pangle, Smaato, Tapjoy, Tenjin, TikTok). The only domain that appears to be first-party for the app publisher is ysocorp.com. Two ysocorp URLs are present: a clean one (https://www.ysocorp.com/privacy-policy) and a clearly malformed one with extra garbage appended. The clean ysocorp URL also appears in applovin_settings.json, which is consistent with providing the app’s own privacy policy to the SDK. Therefore, the APK’s first-party privacy policy URL is https://www.ysocorp.com/privacy-policy.",
  "result": "https://www.ysocorp.com/privacy-policy"
}
{
  "reason": "Most URLs belong to third-party services (ad networks, analytics, SDKs) and must be excluded. The likely first-party developer is YsoCorp (matching the app’s publisher style and appearing in applovin_settings.json). Two YsoCorp URLs are pr

更新表

In [238]:
import re
import shutil
from pathlib import Path
import pandas as pd
from __future__ import annotations
from datetime import datetime

# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch\apk_versions_summary.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary.csv"
maintainess_apk_summary_df = pd.read_csv(apk_summary_df_path)
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                                  apk_name  version  \
 0            com.TunnelSnakes.WrigglySnake       42   
 1            com.TunnelSnakes.WrigglySnake       43   
 2                  com.TwinCrab.Motorpolia      161   
 3                  com.TwinCrab.Motorpolia      164   
 4  com.time.bomb.gun.sound.simulator.prank       37   
 
                                            json_file  source  
 0         com.TunnelSnakes.WrigglySnake-42_urls.json     NaN  
 1         com.TunnelSnakes.WrigglySnake-43_urls.json     NaN  
 2              com.TwinCrab.Motorpolia-161_urls.json     NaN  
 3              com.TwinCrab.Motorpolia-164_urls.json     NaN  
 4  com.time.bomb.gun.sound.simulator.prank-37_url...     NaN  ,
 (100, 4))

In [239]:
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA2_second_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA3_third_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA4_forth_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA5_fifth_100_batch\batch_privacy_results.csv
# 1122apk\1122apk_privacy_policy_url\out_openai_pp_url\AA1_first_batch\AA6_sisth_74_batch\batch_privacy_results.csv
csv_path = Path(rf"1122apk\1122apk_privacy_policy_url\out_openai_pp_url\{f_position}\{s_position}\batch_privacy_results.csv")
batch_privacy_results_df = pd.read_csv(csv_path, dtype={"apk_name": str, "version": str})
batch_privacy_results_df.head(), batch_privacy_results_df.shape

(                                  apk_name version  \
 0            com.TunnelSnakes.WrigglySnake      42   
 1            com.TunnelSnakes.WrigglySnake      43   
 2                  com.TwinCrab.Motorpolia     161   
 3                  com.TwinCrab.Motorpolia     164   
 4  com.time.bomb.gun.sound.simulator.prank      37   
 
                               privacy_url  \
 0  https://www.ysocorp.com/privacy-policy   
 1  https://www.ysocorp.com/privacy-policy   
 2                                     NaN   
 3                                     NaN   
 4  https://bralyvn.com/privacy-policy.php   
 
                                            reasoning  \
 0  Most URLs reference third-party services or ad...   
 1  Most URLs belong to third-party services (ad n...   
 2  All detected URLs belong to third-party servic...   
 3  All detected URLs belong to third-party servic...   
 4  Most URLs are third-party ad/analytics/SDK pri...   
 
                                          sour

In [240]:
# 1. 先确保目标表里有这些列
for col in ["privacy_url", "source_file"]:
    if col not in maintainess_apk_summary_df.columns:
        maintainess_apk_summary_df[col] = pd.NA
maintainess_apk_summary_df.head(), maintainess_apk_summary_df.shape

(                                  apk_name  version  \
 0            com.TunnelSnakes.WrigglySnake       42   
 1            com.TunnelSnakes.WrigglySnake       43   
 2                  com.TwinCrab.Motorpolia      161   
 3                  com.TwinCrab.Motorpolia      164   
 4  com.time.bomb.gun.sound.simulator.prank       37   
 
                                            json_file  source privacy_url  \
 0         com.TunnelSnakes.WrigglySnake-42_urls.json     NaN        <NA>   
 1         com.TunnelSnakes.WrigglySnake-43_urls.json     NaN        <NA>   
 2              com.TwinCrab.Motorpolia-161_urls.json     NaN        <NA>   
 3              com.TwinCrab.Motorpolia-164_urls.json     NaN        <NA>   
 4  com.time.bomb.gun.sound.simulator.prank-37_url...     NaN        <NA>   
 
   source_file  
 0        <NA>  
 1        <NA>  
 2        <NA>  
 3        <NA>  
 4        <NA>  ,
 (100, 6))

In [241]:
# 0. 先复制，避免污染原表
left = maintainess_apk_summary_df.copy()
right = batch_privacy_results_df.copy()
left.head(), right.head()

(                                  apk_name  version  \
 0            com.TunnelSnakes.WrigglySnake       42   
 1            com.TunnelSnakes.WrigglySnake       43   
 2                  com.TwinCrab.Motorpolia      161   
 3                  com.TwinCrab.Motorpolia      164   
 4  com.time.bomb.gun.sound.simulator.prank       37   
 
                                            json_file  source privacy_url  \
 0         com.TunnelSnakes.WrigglySnake-42_urls.json     NaN        <NA>   
 1         com.TunnelSnakes.WrigglySnake-43_urls.json     NaN        <NA>   
 2              com.TwinCrab.Motorpolia-161_urls.json     NaN        <NA>   
 3              com.TwinCrab.Motorpolia-164_urls.json     NaN        <NA>   
 4  com.time.bomb.gun.sound.simulator.prank-37_url...     NaN        <NA>   
 
   source_file  
 0        <NA>  
 1        <NA>  
 2        <NA>  
 3        <NA>  
 4        <NA>  ,
                                   apk_name version  \
 0            com.TunnelSnakes.WrigglySn

In [242]:
# 1. 统一键字段类型
for df in [left, right]:
    df["apk_name"] = df["apk_name"].astype(str).str.strip()
    df["version"] = df["version"].astype(str).str.strip()

In [243]:
# 2. 只保留 privacy_url 有值的行
src = right.loc[
    right["privacy_url"].notna() &
    (right["privacy_url"].astype(str).str.strip() != ""),
    ["apk_name", "version", "privacy_url", "source_file"]
].copy()
src.head(), src.shape

(                                  apk_name version  \
 0            com.TunnelSnakes.WrigglySnake      42   
 1            com.TunnelSnakes.WrigglySnake      43   
 4  com.time.bomb.gun.sound.simulator.prank      37   
 5  com.time.bomb.gun.sound.simulator.prank      41   
 6                         com.time.trigger     346   
 
                               privacy_url  \
 0  https://www.ysocorp.com/privacy-policy   
 1  https://www.ysocorp.com/privacy-policy   
 4  https://bralyvn.com/privacy-policy.php   
 5  https://bralyvn.com/privacy-policy.php   
 6        https://say.games/privacy-policy   
 
                                          source_file  
 0  com.TunnelSnakes.WrigglySnake_42_merged_privac...  
 1  com.TunnelSnakes.WrigglySnake_43_merged_privac...  
 4  com.time.bomb.gun.sound.simulator.prank_37_mer...  
 5  com.time.bomb.gun.sound.simulator.prank_41_mer...  
 6  com.time.trigger_346_merged_privacy_url_eviden...  ,
 (46, 4))

In [244]:
# 4. 左连接到目标表
merged = left.merge(
    src,
    on=["apk_name", "version"],
    how="left",
    suffixes=("", "_new")
)
merged.head(), merged.shape

(                                  apk_name version  \
 0            com.TunnelSnakes.WrigglySnake      42   
 1            com.TunnelSnakes.WrigglySnake      43   
 2                  com.TwinCrab.Motorpolia     161   
 3                  com.TwinCrab.Motorpolia     164   
 4  com.time.bomb.gun.sound.simulator.prank      37   
 
                                            json_file  source privacy_url  \
 0         com.TunnelSnakes.WrigglySnake-42_urls.json     NaN        <NA>   
 1         com.TunnelSnakes.WrigglySnake-43_urls.json     NaN        <NA>   
 2              com.TwinCrab.Motorpolia-161_urls.json     NaN        <NA>   
 3              com.TwinCrab.Motorpolia-164_urls.json     NaN        <NA>   
 4  com.time.bomb.gun.sound.simulator.prank-37_url...     NaN        <NA>   
 
   source_file                         privacy_url_new  \
 0        <NA>  https://www.ysocorp.com/privacy-policy   
 1        <NA>  https://www.ysocorp.com/privacy-policy   
 2        <NA>                

In [245]:
# 6. 更新匹配到的行
mask = merged["privacy_url_new"].notna()
merged.loc[mask, "privacy_url"] = merged.loc[mask, "privacy_url_new"]
merged.loc[mask, "source_file"] = merged.loc[mask, "source_file_new"]

merged["source"] = merged["source"].astype(str).str.strip()
merged.loc[mask, "source"] = "apkitself"

In [246]:
merged.head(), merged.shape

(                                  apk_name version  \
 0            com.TunnelSnakes.WrigglySnake      42   
 1            com.TunnelSnakes.WrigglySnake      43   
 2                  com.TwinCrab.Motorpolia     161   
 3                  com.TwinCrab.Motorpolia     164   
 4  com.time.bomb.gun.sound.simulator.prank      37   
 
                                            json_file     source  \
 0         com.TunnelSnakes.WrigglySnake-42_urls.json  apkitself   
 1         com.TunnelSnakes.WrigglySnake-43_urls.json  apkitself   
 2              com.TwinCrab.Motorpolia-161_urls.json        NaN   
 3              com.TwinCrab.Motorpolia-164_urls.json        NaN   
 4  com.time.bomb.gun.sound.simulator.prank-37_url...  apkitself   
 
                               privacy_url  \
 0  https://www.ysocorp.com/privacy-policy   
 1  https://www.ysocorp.com/privacy-policy   
 2                                    <NA>   
 3                                    <NA>   
 4  https://bralyvn.com/priv

In [247]:
# 7. 删除临时列
merged = merged.drop(columns=["privacy_url_new", "source_file_new"])
merged.head(), merged.shape

(                                  apk_name version  \
 0            com.TunnelSnakes.WrigglySnake      42   
 1            com.TunnelSnakes.WrigglySnake      43   
 2                  com.TwinCrab.Motorpolia     161   
 3                  com.TwinCrab.Motorpolia     164   
 4  com.time.bomb.gun.sound.simulator.prank      37   
 
                                            json_file     source  \
 0         com.TunnelSnakes.WrigglySnake-42_urls.json  apkitself   
 1         com.TunnelSnakes.WrigglySnake-43_urls.json  apkitself   
 2              com.TwinCrab.Motorpolia-161_urls.json        NaN   
 3              com.TwinCrab.Motorpolia-164_urls.json        NaN   
 4  com.time.bomb.gun.sound.simulator.prank-37_url...  apkitself   
 
                               privacy_url  \
 0  https://www.ysocorp.com/privacy-policy   
 1  https://www.ysocorp.com/privacy-policy   
 2                                    <NA>   
 3                                    <NA>   
 4  https://bralyvn.com/priv

In [248]:
# 8. 保存回 CSV
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch\apk_versions_summary_gpt.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch\apk_versions_summary_gpt.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch\apk_versions_summary_gpt.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch\apk_versions_summary_gpt.csv
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_74_batch\apk_versions_summary_gpt.csv
apk_summary_df_path = rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}\apk_versions_summary_gpt.csv"
merged.to_csv(apk_summary_df_path, index=False)